# Kalakar TFT — Colab GPU Trainer

Runs **Step 1** (retrain to Dec 2021) and **Step 5** (XGB-as-TFT-feature fusion) on a free Colab T4 in roughly 10–20 min each. Produces two new checkpoints that you drop into your local `models/` folder.

### One-time setup
1. Runtime → Change runtime type → **T4 GPU**.
2. Upload this notebook + a zip of the project folder (the `kalakar_2/` directory). The easy way: on your laptop `cd ..; 7z a kalakar_2.zip kalakar_2` or right-click → Send to → compressed folder, then drag-drop into Colab's left file panel.
3. Set `PROJECT_ZIP` below to the uploaded filename and click **Runtime → Run all**.
4. Two new `.ckpt` files appear in `/content/kalakar_2/models/`. Download them. Drop into your local `models/` folder. Done.

In [ ]:
PROJECT_ZIP = "kalakar_2.zip"   # <-- change if your zip has a different name

import os, zipfile, subprocess, sys

if not os.path.exists('/content/kalakar_2'):
    assert os.path.exists(f'/content/{PROJECT_ZIP}'), f'Upload {PROJECT_ZIP} to /content/ first'
    with zipfile.ZipFile(f'/content/{PROJECT_ZIP}') as z:
        z.extractall('/content/')

os.chdir('/content/kalakar_2')
print('CWD:', os.getcwd())
print('Files:', sorted(os.listdir('.'))[:15])

In [ ]:
# Install dependencies (Colab already has torch w/ CUDA — we just add forecasting libs)
!pip -q install pytorch-forecasting==1.7.0 pytorch-lightning==2.6.1 pytorch_optimizer joblib scikit-learn

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## Step 1 — retrain TFT to Dec 2021

In [ ]:
!python scripts/09_retrain_tft_2022.py --gpus 1 --epochs 30 --batch_size 128

## Step 5 — XGB-as-TFT-feature fusion

In [ ]:
!python scripts/10_xgb_as_tft_feature.py --gpus 1 --epochs 30 --batch_size 128

## Package checkpoints for download

In [ ]:
import shutil, glob
out = '/content/kalakar_trained.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob('models/tft_best_2022*.ckpt') + glob.glob('models/tft_best_xgbfused*.ckpt') \
            + glob.glob('models/xgb_clean_2019.pkl') + glob.glob('models/tft_config_*.json') \
            + glob.glob('data/processed/master_dataset_xgbfused.csv'):
        z.write(p)
        print('added', p)
print(f'\nZip: {out}  size={os.path.getsize(out)/1e6:.1f} MB')
print('Download this file, unzip, and merge the files back into your local kalakar_2/ folder.')

In [ ]:
from google.colab import files
files.download('/content/kalakar_trained.zip')